# Buffers and Selecting by Location

**Series:** Geoprocessing in Python (notebook 1 of 3)
**Time:** about 30 minutes

**In this notebook you will:**
- Put every layer in a projected CRS before measuring
- Draw buffers around points and dissolve them into one zone
- Select features by their location: intersects, within, and centroid within
- See what goes wrong when you buffer in degrees

**You should already know:** GeoDataFrames, CRS and joins (Foundations notebooks <a href="https://colab.research.google.com/github/SpatialTurn/Foundations/blob/main/notebooks/04a_Vector_Data_with_GeoPandas.ipynb">4a</a> to <a href="https://colab.research.google.com/github/SpatialTurn/Foundations/blob/main/notebooks/04c_Joining_Tables_and_Layers.ipynb">4c</a>).

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## The question

Which parts of Lafayette and West Lafayette are within **1 mile (about 1,609 meters)** of a supermarket, as
the crow flies? One mile is a common walking threshold in food access research. We answer it with three
everyday tools: **buffer**, **dissolve** and **select by location**.

We use three layers, all read straight from the web:

| Layer | Contents |
|---|---|
| `grocery_stores` | 64 food stores from OpenStreetMap, with a `store_type` column (Supermarket, Convenience, Wholesale) |
| `study_area` | The Lafayette and West Lafayette city limits |
| `tippecanoe_tracts` + `tippecanoe_acs_2024.csv` | 43 census tracts, joined to 2024 ACS estimates by `GEOID` |

In [ ]:
import geopandas as gpd
import pandas as pd

stores = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/CaseStudies/main/data/grocery_stores.geojson")
city = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/CaseStudies/main/data/study_area.geojson")
tracts = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_tracts.geojson")
acs = pd.read_csv("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_acs_2024.csv", dtype={"GEOID": str})   # keep GEOID as text

print(stores.crs, city.crs, tracts.crs)
stores["store_type"].value_counts()

## Step 1: Project to meters

All three layers are in **EPSG:4326** (degrees). A buffer distance is in the layer's units, so we first convert
everything to **EPSG:26916** (UTM zone 16N, meters). We also join the ACS table to the tracts, as in Foundations
notebook 4c.

In [ ]:
stores = stores.to_crs(epsg=26916)
city = city.to_crs(epsg=26916)
tracts = tracts.to_crs(epsg=26916).merge(acs.drop(columns="name"), on="GEOID")

print(stores.crs.name, "|", len(tracts), "tracts after the join")
supermarkets = stores[stores["store_type"] == "Supermarket"]
print(len(supermarkets), "supermarkets")

## Step 2: Buffer

`buffer(distance)` draws a polygon around every feature at the given distance. Around a point it is a circle.

In [ ]:
buffers = supermarkets.copy()
buffers["geometry"] = supermarkets.buffer(1609)      # 1 mile in meters

ax = city.plot(color="whitesmoke", edgecolor="gray", figsize=(7, 7))
buffers.plot(ax=ax, color="tab:green", alpha=0.3, edgecolor="tab:green")
supermarkets.plot(ax=ax, color="black", markersize=10)
ax.set_title("1-mile buffers around supermarkets")
ax.set_axis_off()

## What happens in degrees?

Here is the same buffer on the **unprojected** layer. GeoPandas warns you, and for good reason: the distance
is read as 1,609 **degrees**, so every "circle" is larger than the whole planet.

In [ ]:
wrong = supermarkets.to_crs(epsg=4326).buffer(1609)
print(wrong.iloc[0].bounds)    # (min lon, min lat, max lon, max lat): far beyond -180 to 180

Even a sensible-looking degree buffer such as `buffer(0.015)` is wrong: a degree of longitude in Indiana is
about 85 km but a degree of latitude is about 111 km, so the "circle" is really an ellipse. **Always project
before buffering.**

## Step 3: Dissolve

Supermarkets near each other have overlapping buffers. If we added up the buffer areas we would count the
overlaps twice. **Dissolving** merges all the buffers into one zone.

In [ ]:
print("Sum of buffer areas:", round(buffers.area.sum() / 1e6, 1), "km2")

near = buffers.dissolve()          # one row, one (multi)polygon
print("Dissolved area:     ", round(near.area.sum() / 1e6, 1), "km2")

## Step 4: Select by location

A **spatial predicate** asks a yes or no question about two shapes. The most useful ones:

| Predicate | True when |
|---|---|
| `intersects` | The shapes touch or overlap at all |
| `within` | The first shape is completely inside the second |
| `contains` | The second shape is completely inside the first |

Which tracts are "near" a supermarket depends on the predicate you choose.

In [ ]:
zone = near.geometry.iloc[0]

touching = tracts[tracts.intersects(zone)]
inside = tracts[tracts.within(zone)]
centroid_in = tracts[tracts.centroid.within(zone)]

print("Tracts that touch the zone:        ", len(touching))
print("Tracts completely inside the zone: ", len(inside))
print("Tracts whose center is in the zone:", len(centroid_in))

In [ ]:
ax = tracts.plot(color="whitesmoke", edgecolor="gray", figsize=(7, 7))
centroid_in.plot(ax=ax, color="tab:green", alpha=0.6)
near.boundary.plot(ax=ax, color="black")
xmin, ymin, xmax, ymax = city.total_bounds      # zoom to the cities
ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax)
ax.set_title("Tracts whose center is within 1 mile of a supermarket")
ax.set_axis_off()

None of these is "right". `intersects` is generous (a tract counts if one corner touches the zone), `within` is
strict, and the centroid test is a common compromise. Whichever you choose, **say which one you used**, because
it changes the answer. Notebook 01b avoids the problem by measuring how much of each tract is in the zone.

Select by location also works between point layers and polygons, for example to count stores inside an area.

In [ ]:
convenience = stores[stores["store_type"] == "Convenience"]
close = convenience[convenience.within(zone)]
print(len(close), "of", len(convenience), "convenience stores are within 1 mile of a supermarket")

## Try it yourself

Repeat the analysis with a **half-mile** (805 m) buffer, which is closer to a 10-minute walk.

1. Buffer the supermarkets by 805 m and dissolve.
2. How many tracts have their center inside the half-mile zone?

*Hint:* Copy Steps 2 to 4 and change the distance.

<details>
<summary><b>Show a solution</b></summary>

```python
half = supermarkets.copy()
half["geometry"] = supermarkets.buffer(805)
half_zone = half.dissolve().geometry.iloc[0]
print(tracts.centroid.within(half_zone).sum(), "tracts")
```

</details>

In [ ]:
# Write your answer here

## Recap

- Project to a CRS in meters (here EPSG:26916) before any buffer or distance
- `buffer()` draws zones; `dissolve()` merges overlapping zones so areas are not double counted
- Spatial predicates (`intersects`, `within`, centroid `within`) select by location, and the choice changes the result

**Next:** [01b Clip, Intersect and Dissolve](https://colab.research.google.com/github/SpatialTurn/DataAnalysis/blob/main/notebooks/01b_Clip_Intersect_and_Dissolve.ipynb)